In [ ]:
# import Pkg
# Pkg.add("Plots")
# Pkg.add("MultiObjectiveAlgorithms")
# Pkg.add("MathOptInterface")
# Pkg.add("Printf")

In [ ]:
ENV["OMP_NUM_THREADS"] = "14"

In [ ]:
using JuMP
# using HiGHS
using SCIP
# using EzXML
using Plots
# using Ipopt
# using Cbc
import MultiObjectiveAlgorithms as MOA
import MathOptInterface as MOI
using Printf
using PrettyTables

In [ ]:
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7

In [ ]:
function read_input(file_path::String, validate::Bool)
    lines = readlines(file_path)

    lines = filter(line -> !isempty(line) && !startswith(strip(line), "#"), lines)
    lines = map(line -> split(line, "#")[1] |> strip, lines)

    idx = 1

    D = parse(Int, lines[idx]); idx += 1
    D_ = parse.(Int, split(lines[idx])); idx += 1
    D_star = parse.(Int, split(lines[idx])); idx += 1
    P = parse(Int, lines[idx]); idx += 1
        Player_names = split(lines[idx]); idx += 1
    S = parse(Int, lines[idx]); idx += 1
        Skill_names = split(lines[idx]); idx += 1
    T = parse(Int, lines[idx]); idx += 1
        Tactic_names = split(lines[idx]); idx += 1
    E = parse(Int, lines[idx]); idx += 1
        Exercise_names = split(lines[idx]); idx += 1

    a0 = parse.(Int, split(lines[idx])); idx += 1
    b0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
        c0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
        h = [parse.(Int, split(lines[idx + i - 1])) for i in 1:P]; idx += P
        w = parse.(Int, split(lines[idx])); idx += 1
        b = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
        c = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
        v = parse.(Int, split(lines[idx])); idx += 1
        V = parse.(Int, split(lines[idx])); idx += 1

        B_star = Array{Int, 3}(undef, S, P, length(D_star))
        for d in 1:length(D_star)
        for s in 1:S
            B_star[s, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

        C_star = Array{Int, 3}(undef, T, P, length(D_star))
        for d in 1:length(D_star)
        for t in 1:T
            C_star[t, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

        if validate==true
                # input validation


        end

        return D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star
end

In [ ]:
function δ1(d_::Int, d::Int)
	return k1*ℯ^(-(d-d_)/τ1)
end

function δ2(d_::Int, d::Int)
	return k2*ℯ^(-(d-d_)/τ2)
end

function β(s::Int, p::Int, d::Int)
	return 1
end

function γ(t::Int, d::Int)
	return 1
end

function ρ(t::Int)
	return 2
end

In [ ]:
# Get input parameters from .txt file
input_files_folder = "input"
# input_file_name = "input_smaller.txt"
input_file_name = "input_current.txt"
file_path = input_files_folder * "/" * input_file_name
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

In [ ]:
# Define model
# model = Model(HiGHS.Optimizer)
model = Model(SCIP.Optimizer)
# model = Model(() -> MOA.Optimizer(
#         optimizer_with_attributes(
#         HiGHS.Optimizer,
#         "output_flag" => true,
#         "mip_rel_gap" => 0.05,
#         "threads" => 8
#         )
# ))

In [ ]:
# Define variables
@variable(model, x[1:E, 1:length(D_)] >= 0, Int)

In [ ]:
@variable(model, y[1:S, 1:P, 1:length(D_star)], Bin)

In [ ]:
@variable(model, z[1:T, 1:P, 1:length(D_star)], Bin)

In [ ]:
@variable(model, Z[1:T, 1:length(D_star)], Bin)

In [ ]:
@variable(model, A_min[1:length(D_star)])

In [ ]:
# Вспомогательные функции и компоненты целевой функции
function A(p::Int, d::Int)
	result = a0[p]
	result += sum(δ1(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	result -= sum(δ2(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function A_avg(d::Int)
	return sum(A(p, d) for p in 1:P)/P
end

α = 0.5
function A()
	return sum(α*A_min[d_]+(1-α)*A_avg(D_star[d_]) for d_ in 1:length(D_star))
end

function B(s::Int, p::Int, d::Int)
	result = b0[s][p]
	result += sum(h[p][d_]*sum(b[s][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function B()
	return sum(sum(sum(β(s, p, d_)*y[s, p, d_] for s in 1:S) for p in 1:P) for d_ in 1:length(D_star))
end

function C(t::Int, p::Int, d::Int)
	result = c0[t][p]
	result += sum(h[p][d_]*sum(c[t][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
end

function C()
	return sum(sum(γ(t, d_)*Z[t, d_] for t in 1:T) for d_ in 1:length(D_star))
end

In [ ]:
# Define constraints
@constraint(model, [d_=1:length(D_)], sum(v[e]*x[e, d_] for e in 1:E) <= V[d_])

In [ ]:
@constraint(model, [s=1:S, p=1:P, d_=1:length(D_star)], B(s, p, D_star[d_]) + B_star[s, p, d_]*(1-y[s, p, d_]) >= B_star[s, p, d_])

In [ ]:
@constraint(model, [t=1:T, p=1:P, d_=1:length(D_star)], C(t, p, D_star[d_]) + C_star[t, p, d_]*(1-z[t, p, d_]) >= C_star[t, p, d_])

In [ ]:
@constraint(model, [t=1:T, d_=1:length(D_star)], sum(z[t, p, d_] for p in 1:P) >= ρ(t)*Z[t, d_])

In [ ]:
@constraint(model, [d_=1:length(D_star), p=1:P], A_min[d_] <= A(p, D_star[d_]))

In [ ]:
# Get ideal point
@objective(model, Max, A())

# HiGHS
# set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
# set_optimizer_attribute(model, "mip_rel_gap", 0.01)
# set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes

# SCIP
set_optimizer_attribute(model, "limits/gap", 0.01)
set_optimizer_attribute(model, "limits/time", 1200.0) # 20 minutes

optimize!(model)
A_optimal = value(A())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

In [ ]:
@objective(model, Max, B())

# HiGHS
# set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
# set_optimizer_attribute(model, "mip_rel_gap", 0.01)
# set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes

# SCIP
set_optimizer_attribute(model, "limits/gap", 0.05)
set_optimizer_attribute(model, "limits/time", 14400.0) # 20 minutes

optimize!(model)
B_optimal = value(B())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

In [ ]:
@objective(model, Max, C())

# HiGHS
# set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
# set_optimizer_attribute(model, "mip_rel_gap", 0.01)
# set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes

# SCIP
set_optimizer_attribute(model, "limits/gap", 0.01)
set_optimizer_attribute(model, "limits/time", 1200.0) # 20 minutes

optimize!(model)
C_optimal = value(C())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

In [ ]:
ideal_point = [A_optimal, B_optimal, C_optimal]

In [ ]:
A_norm = () -> A()/A_optimal
B_norm = () -> B()/B_optimal
C_norm = () -> C()/C_optimal

In [ ]:
# Create an array of weights
weights_for_moa =   [[0.33, 0.33, 0.33],
                     [0.5, 0.25, 0.25],
                     [0.25, 0.5, 0.25],
                     [0.25, 0.25, 0.5],
                    ]

In [ ]:
# Solve model with different weight distributions using warm-start (progressive gap tightening)
set_silent(model)

n_passes = 2                      # warm-start passes per weight combination
gaps = [0.10, 0.03]         # progressively tighter MIP relative gaps
time_limits = [3600.0, 10800.0]

output_folder = "results"
mkpath(output_folder)
solutions = []
solutions_nonrm = []
solution_weights = []   # weights for each successful solution (for visualization)

for i in 1:length(weights_for_moa)
    wA, wB, wC = weights_for_moa[i]
    @objective(model, Max, wA*A_norm() + wB*B_norm() + wC*C_norm())

    println("Solving weight combination $i/$(length(weights_for_moa)): wA=$wA, wB=$wB, wC=$wC")

    # best_sol stores all values extracted right after optimize!(), before any model modification
    best_sol = nothing

    for pass in 1:n_passes

		# HiGHS
        # set_optimizer_attribute(model, "mip_rel_gap", gaps[pass])
		# set_optimizer_attribute(model, "time_limit", time_limits[pass]) # 20 minutes

		# SCIP
		set_optimizer_attribute(model, "limits/gap", gaps[pass])
		set_optimizer_attribute(model, "limits/time", time_limits[pass]) # 20 minutes

        optimize!(model)

        if primal_status(model) != MOI.FEASIBLE_POINT
            println("  Pass $pass: no feasible solution (status=$(termination_status(model)))")
            break
        end

        println("  Pass $pass (gap=$(gaps[pass])): obj = $(round(objective_value(model), digits=6))")
		if termination_status(model) == TIME_LIMIT
			gap = relative_gap(model)
			println("Relative gap: $(gap * 100)%")
		end

        # ── Extract ALL values immediately, before any model modification ──
        x_vals     = round.(Int, value.(x))
        y_vals     = round.(Int, value.(y))
        z_vals     = round.(Int, value.(z))
        Z_vals     = round.(Int, value.(Z))
        A_min_vals = value.(A_min)
        # Compute objective components while model is still in solved state
        A_val      = value(A())
        B_val      = value(B())
        C_val      = value(C())
        An_val     = value(A_norm())
        Bn_val     = value(B_norm())
        Cn_val     = value(C_norm())

        best_sol = (x=x_vals, y=y_vals, z=z_vals, Z=Z_vals, A_min=A_min_vals,
                    A=A_val, B=B_val, C=C_val,
                    A_norm=An_val, B_norm=Bn_val, C_norm=Cn_val)

        # ── Now safe to modify model for warm-start ──
        if pass < n_passes
            set_start_value.(x,     x_vals)
            set_start_value.(y,     y_vals)
            set_start_value.(z,     z_vals)
            set_start_value.(Z,     Z_vals)
            set_start_value.(A_min, A_min_vals)
        end
    end

    if best_sol === nothing
        println("  Weight combination $i: no feasible solution found, skipping.\n")
        continue
    end

    push!(solutions,        [best_sol.A_norm, best_sol.B_norm, best_sol.C_norm])
    push!(solutions_nonrm,  [best_sol.A,      best_sol.B,      best_sol.C])
    push!(solution_weights, [wA, wB, wC])

    output_file = "$(input_file_name)_$(weights_for_moa[i])"

    open(output_folder * "/" * output_file, "w") do io
        println(io, "Solution of model with weight distribution ", weights_for_moa[i])
        println(io, "Ideal Point (Bound): ", ideal_point)

        println(io, "---Итоговая эффективность тренировок---")
        println(io, "A - общая физическая готовность команды\nB - бонус за освоенные индивидуальные навыки\nC - бонус за освоенные командой тактики\n")
        println(io, "A = $(best_sol.A)\nB = $(best_sol.B)\nC = $(best_sol.C)\n\n")
        println(io, "---Итоговая эффективность тренировок относительно идеальной точки---")
        println(io, "A = $(best_sol.A_norm)\nB = $(best_sol.B_norm)\nC = $(best_sol.C_norm)\n\n")

        println(io, "---Расписание тренировок---\n")
        n_days = length(D_)
        data = Array{Any}(undef, n_days, 2)
        for d_ in 1:n_days
            data[d_, 1] = "Тренировка №$(d_) (день №$(D_[d_]))"
            day_exercises = String[]
            for e in 1:E
                if best_sol.x[e, d_] >= 1
                    push!(day_exercises, Exercise_names[e])
                end
            end
            data[d_, 2] = join(day_exercises, ", ")
        end
        pretty_table(io, data, crop = :none)

        println(io, "---Освоенные навыки---\n")
        for p in 1:P
            for d_ in 1:length(D_star)
                println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
                learned_anything = false
                for s in 1:S
                    if best_sol.y[s, p, d_] == 1 && (d_ == 1 || best_sol.y[s, p, d_-1] != 1)
                        learned_anything = true
                        println(io, "Освоил навык \'$(Skill_names[s])\'")
                    end
                end
                if !learned_anything; println(io, "Ничего не освоил"); end
            end
            print(io, "\n")
        end

        println(io, "---Освоенные индивидуально тактики---\n")
        for p in 1:P
            for d_ in 1:length(D_star)
                println(io, "Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
                learned_anything = false
                for t in 1:T
                    if best_sol.z[t, p, d_] == 1 && (d_ == 1 || best_sol.z[t, p, d_-1] != 1)
                        learned_anything = true
                        println(io, "Освоил тактику \'$(Tactic_names[t])\'")
                    end
                end
                if !learned_anything; println(io, "Ничего не освоил"); end
            end
            print(io, "\n")
        end

        println(io, "---Освоенные командой тактики---\n")
        for d_ in 1:length(D_star)
            for t in 1:T
                println(io, "Тактика: $(Tactic_names[t]), контрольная дата №$(d_) (день $(D_star[d_]))")
                println(io, best_sol.Z[t, d_] == 1 ? "Освоено" : "Не освоено")
            end
            print(io, "\n")
        end
    end

    println()
end


In [ ]:
# Write visualization data to txt file for Python visualizer
viz_file = output_folder * "/" * input_file_name * "_viz_data.txt"
open(viz_file, "w") do io
    println(io, "# Multi-criteria optimization visualization data")
    println(io, "# ideal_point,A,B,C")
    println(io, "# solution,wA,wB,wC,A,B,C,A_norm,B_norm,C_norm")
    println(io, "ideal_point,$(ideal_point[1]),$(ideal_point[2]),$(ideal_point[3])")
    for i in 1:length(solutions)
        wA, wB, wC = solution_weights[i]
        A_val, B_val, C_val = solutions_nonrm[i]
        An, Bn, Cn = solutions[i]
        println(io, "solution,$wA,$wB,$wC,$A_val,$B_val,$C_val,$An,$Bn,$Cn")
    end
end
println("Visualization data written to: $viz_file")
